*Basic SDM*

In [ ]:
!pip install diffusers

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)
pipe = pipe.to("cuda")


In [ ]:
prompt = "a snowy mountain under a shining moon"
image = pipe(prompt).images[0]

In [ ]:
# import torch
# from diffusers import StableDiffusionPipeline, DDIMScheduler
from PIL import Image
import matplotlib.pyplot as plt

# # Load the pipeline
# pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)
# pipe = pipe.to("cuda")

# # Switch to DDIMScheduler for finer control over steps
# pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

# # Define prompt
# prompt = "a photo of an astronaut riding a horse on mars"

# List to store intermediate images
saved_images = []

# Custom callback function to save images during inference
def save_intermediate_images(step, timestep, latents):
    if step % 2 == 0:  # Save every K steps
        with torch.no_grad():
            # Decode the latents to an image
            image = pipe.decode_latents(latents)
            pil_image = pipe.numpy_to_pil(image)[0]

            # Save to list
            saved_images.append(pil_image)

            # Display the image
            plt.imshow(pil_image)
            plt.axis("off")
            plt.title(f"Step: {step}")
            plt.show()

# Generate the image with intermediate steps
image = pipe(
    prompt,
    num_inference_steps=50,  # Total steps
    guidance_scale=7.5,  # Optional, for classifier-free guidance
    callback=save_intermediate_images,  # Add callback
    callback_steps=1,  # Frequency of calling the callback
).images[0]

# Display final image
plt.imshow(image)
plt.axis("off")
plt.title("Final Image")
plt.show()

# Saved images are stored in the `saved_images` list


In [ ]:
from skimage.segmentation import slic,quickshift,felzenszwalb
import numpy as np

segments_1 = slic(np.array(image), n_segments=100, compactness=10)
segments_2 = quickshift(np.array(image), kernel_size=3, max_dist=60, ratio=0.2)
segments_3 = felzenszwalb(np.array(image), scale=100, sigma=0.5, min_size=50)

In [ ]:
def segment_image_felzenszwalb(image, scale=100, sigma=0.5, min_size=50):
    """
    Segments an image using Felzenszwalb's algorithm and returns a list of images,
    where each image retains only one segment while the rest is set to zero.

    :param image: Input image as a NumPy array (H, W, C)
    :param scale: Scale parameter for Felzenszwalb segmentation
    :param sigma: Width of Gaussian kernel for preprocessing
    :param min_size: Minimum component size
    :return: List of segmented images
    """
    # Convert to float32 if not already
    image_float = image.astype(np.float32) / 255.0 if image.dtype != np.float32 else image

    # Apply Felzenszwalb segmentation
    segments = felzenszwalb(image_float, scale=scale, sigma=sigma, min_size=min_size)
    unique_segments = np.unique(segments)

    # Create segmented images
    segmented_images = []
    for segment in unique_segments:
        mask = (segments == segment).astype(np.uint8)
        masked_image = image * mask[:, :, np.newaxis]  # Retain color where mask is 1
        segmented_images.append(masked_image)

    return segmented_images

In [ ]:
segmented_images = segment_image_felzenszwalb(np.array(image),scale = 1100)

In [ ]:
len(
    segmented_images
)

In [ ]:
count = 0
for seg in segmented_images:
    if count == 60:
      break
    plt.imshow(seg)
    plt.axis("off")
    plt.show()
    count = count + 1

In [ ]:
fig, axes = plt.subplots(1, len(segmented_images), figsize=(100, 100))
for ax, seg_img in zip(axes, segmented_images):
        ax.imshow(seg_img)
        ax.axis("off")
plt.show()

In [ ]:
# Display the image
plt.imshow(segments_3)
plt.axis("off")
plt.show()

In [ ]:
from sklearn.cluster import KMeans
import cv2
import numpy as np
from PIL import Image

def compute_dominant_colors(image, k=5, color_space='RGB'):
    # Convert PIL.Image.Image to NumPy array (OpenCV uses BGR format by default)
    image = np.array(image)
    if image.ndim == 2:  # Handle grayscale images
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    # Ensure the image is in BGR format for OpenCV processing
    if image.shape[-1] == 4:  # Handle RGBA images
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2BGR)
    elif image.shape[-1] == 3 and color_space == 'RGB':
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # Convert to specified color space
    if color_space == 'HSV':
        image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    elif color_space == 'Lab':
        image = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
    else:  # Default to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Reshape image into 2D array of pixels
    pixels = image.reshape(-1, 3)

    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(pixels)

    # Get dominant colors and proportions
    dominant_colors = kmeans.cluster_centers_
    proportions = np.bincount(kmeans.labels_) / len(kmeans.labels_)

    return dominant_colors, proportions



In [ ]:
metric1 = []

for img in saved_images:
  #print(type(img))
      metric1.append(compute_dominant_colors(img))

In [ ]:
len(metric1)

In [ ]:
for index in range(len(metric1)):
   print("timestep", index*2  ,"---------")
   print(metric1[index][0])
   print("\n")
   print(metric1[index][1])
   print("---------")


*DC based segmentation*

In [ ]:
'''
A new array of pixels is created, where each pixel is replaced by its cluster's dominant color.
The resulting array has the same number of pixels as the original image, but each pixel now represents its cluster's dominant color.
'''

In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans
from matplotlib import pyplot as plt
from PIL import Image

def compute_dominant_colors(image, k=12, color_space='RGB', save_path=None):
    # Convert PIL.Image.Image to NumPy array (OpenCV uses BGR format by default)
    image = np.array(image)
    if image.ndim == 2:  # Handle grayscale images
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    # Ensure the image is in BGR format for OpenCV processing
    if image.shape[-1] == 4:  # Handle RGBA images
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2BGR)
    elif image.shape[-1] == 3 and color_space == 'RGB':
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # Convert to specified color space
    if color_space == 'HSV':
        image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    elif color_space == 'Lab':
        image = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
    else:  # Default to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Reshape image into 2D array of pixels
    pixels = image.reshape(-1, 3)

    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(pixels)

    # Get dominant colors and proportions
    dominant_colors = kmeans.cluster_centers_
    proportions = np.bincount(kmeans.labels_) / len(kmeans.labels_)

    # Map each pixel to its cluster's dominant color
    segmented_image = np.array([dominant_colors[label] for label in kmeans.labels_], dtype=np.uint8)
    segmented_image = segmented_image.reshape(image.shape)

    # Plot original image and segmented image
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Segmented Image")
    plt.imshow(cv2.cvtColor(segmented_image, cv2.COLOR_BGR2RGB if color_space == 'RGB' else cv2.COLOR_HSV2RGB))
    plt.axis("off")

    plt.tight_layout()

    # Save the plot as an image
    if save_path:
        plt.savefig(save_path, format='png', bbox_inches='tight')
    plt.show()

    return dominant_colors, proportions


In [ ]:
for img in saved_images:
  compute_dominant_colors(img, k=12)

In [ ]:
for img in saved_images:
  compute_dominant_colors(img, k=5)

In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans
from matplotlib import pyplot as plt
from PIL import Image

def compute_and_display_segments(image, k=5, color_space='RGB', save_path=None, image_number=None):
    # Convert PIL.Image.Image to NumPy array (OpenCV uses BGR format by default)
    image = np.array(image)
    if image.ndim == 2:  # Handle grayscale images
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    # Ensure the image is in BGR format for OpenCV processing
    if image.shape[-1] == 4:  # Handle RGBA images
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2BGR)
    elif image.shape[-1] == 3 and color_space == 'RGB':
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # Convert to specified color space
    if color_space == 'HSV':
        image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    elif color_space == 'Lab':
        image = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
    else:  # Default to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Reshape image into 2D array of pixels
    pixels = image.reshape(-1, 3)

    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(pixels)

    # Get cluster labels and dominant colors
    labels = kmeans.labels_
    dominant_colors = kmeans.cluster_centers_

    # Create segmented images for each cluster
    segmented_images = []
    for cluster_idx in range(k):
        # Create a mask for the current cluster
        mask = (labels == cluster_idx)

        # Initialize a black image
        cluster_image = np.zeros_like(pixels, dtype=np.uint8)

        # Set pixels belonging to the cluster to their original color
        cluster_image[mask] = pixels[mask]

        # Reshape to original dimensions
        cluster_image = cluster_image.reshape(image.shape)
        segmented_images.append(cluster_image)

    # Plot the original image and each segment
    plt.figure(figsize=(15, 8))
    plt.subplot(2, (k + 1) // 2 + 1, 1)
    plt.title("Original Image")
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    for i, segmented in enumerate(segmented_images, start=2):
        plt.subplot(2, (k + 1) // 2 + 1, i)
        plt.title(f"Segment {i - 1}")
        plt.imshow(cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB))
        plt.axis("off")

    plt.tight_layout()

    # Save the plot if required
    if save_path and image_number is not None:
        dynamic_save_path = f"{save_path}_image_{image_number}.png"
        plt.savefig(dynamic_save_path, format='png', bbox_inches='tight')
        print(f"Saved the combined plot to {dynamic_save_path}")

    plt.show()


In [ ]:
for img in saved_images:
  compute_and_display_segments(img, k=5)

In [ ]:
for img in saved_images:
  compute_and_display_segments(img, k=12)

*after LoRa fine tuning*